## Setting paths and importing 

In [1]:
from pathlib import Path
import json
import re
import sys
import unicodedata

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import balanced_accuracy_score, silhouette_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, classification_report

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
from src.config import PROJECT_ROOT, ensure_default_directories
from src.utils import clean_country_code
from src.validation import assert_exported, assert_non_empty, assert_required_columns

ensure_default_directories()
FINAL_ANALYSIS_DIR = PROJECT_ROOT / "7 Final Analysis"
COUNTRY_DIR = PROJECT_ROOT / "6 Country level variables"
LLM_WIKIPEDIA_DIR = PROJECT_ROOT / "5 LLM Wikipedia analysis"
OUTPUTS_DIR = FINAL_ANALYSIS_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DEMOGRAPHY_PATH_CANDIDATES = [
    COUNTRY_DIR / "outputs" / "tables" / "demography_country_year.csv",
    COUNTRY_DIR / "demography_country_year.csv",
]

EXTRACTED_POLARISATION_PATH_CANDIDATES = [
    LLM_WIKIPEDIA_DIR / "outputs" / "tables" / "extracted_polarisation_final.xlsx",
    LLM_WIKIPEDIA_DIR / "extracted_polarisation_final.xlsx",
]

DEMOGRAPHY_PATH = next((p for p in DEMOGRAPHY_PATH_CANDIDATES if p.exists()), DEMOGRAPHY_PATH_CANDIDATES[0])
EXTRACTED_POLARISATION_PATH = next((p for p in EXTRACTED_POLARISATION_PATH_CANDIDATES if p.exists()), EXTRACTED_POLARISATION_PATH_CANDIDATES[0])

MERGED_OUTPUT_PATH = TABLES_DIR / "polarisation_with_demography.xlsx"
CLEANED_DATASET_PATH = TABLES_DIR / "cleaned_mep_analysis_dataset.csv"

assert DEMOGRAPHY_PATH.exists(), f"Demography file missing: {DEMOGRAPHY_PATH}"
assert EXTRACTED_POLARISATION_PATH.exists(), f"Extraction file missing: {EXTRACTED_POLARISATION_PATH}"
assert CLEANED_DATASET_PATH.exists(), f"Cleaned dataset file missing: {CLEANED_DATASET_PATH}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Demography file: {DEMOGRAPHY_PATH}")
print(f"Extraction file: {EXTRACTED_POLARISATION_PATH}")
print(f"Tables dir: {TABLES_DIR}")

Project root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics
Demography file: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\6 Country level variables\outputs\tables\demography_country_year.csv
Extraction file: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\5 LLM Wikipedia analysis\outputs\tables\extracted_polarisation_final.xlsx
Tables dir: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\7 Final Analysis\outputs\tables


## Load in cleaned file for analysis

In [3]:
# ============================================================
# Merge each MEP with average 2010-2025 country-level variables
# Output: final_analysis.csv
# ============================================================

cleaned_df = pd.read_csv(CLEANED_DATASET_PATH)
demography_df = pd.read_csv(DEMOGRAPHY_PATH)

cleaned_df = cleaned_df.copy()
demography_df = demography_df.copy()

# ------------------------------------------------------------
# Create country keys
# ------------------------------------------------------------

cleaned_df["country_key"] = cleaned_df["country_clean_final"].apply(clean_country_code)

if "country" in demography_df.columns:
    demography_df["country_key"] = demography_df["country"].apply(clean_country_code)
elif "Country" in demography_df.columns:
    demography_df["country_key"] = demography_df["Country"].apply(clean_country_code)
elif "country_code" in demography_df.columns:
    demography_df["country_key"] = demography_df["country_code"].apply(clean_country_code)
else:
    raise ValueError("No country column found in demography_df.")

if "year" not in demography_df.columns:
    raise ValueError("No year column found in demography_df.")

# ------------------------------------------------------------
# Keep 2010-2025 only
# ------------------------------------------------------------

demography_period = demography_df.loc[
    demography_df["year"].between(2010, 2025)
].copy()

# ------------------------------------------------------------
# Average numeric country variables by country
# Missing values are ignored by default in mean()
# ------------------------------------------------------------

numeric_demo_cols = demography_period.select_dtypes(include=["number"]).columns.tolist()

# Do not average year itself
numeric_demo_cols = [c for c in numeric_demo_cols if c != "year"]

demography_avg = (
    demography_period
    .groupby("country_key", as_index=False)[numeric_demo_cols]
    .mean()
)

# Add metadata: how many years contributed per country
demography_avg["demography_years_available_2010_2025"] = (
    demography_period
    .groupby("country_key")["year"]
    .nunique()
    .reindex(demography_avg["country_key"])
    .values
)

# Rename averaged columns clearly
demography_avg = demography_avg.rename(
    columns={
        col: f"{col}_avg_2010_2025"
        for col in numeric_demo_cols
    }
)

print("Country-level averaged demography:", demography_avg.shape)

# ------------------------------------------------------------
# Check unmatched countries before merge
# ------------------------------------------------------------

mep_countries = set(cleaned_df["country_key"].dropna().unique())
demo_countries = set(demography_avg["country_key"].dropna().unique())

missing_in_demo = sorted(mep_countries - demo_countries)

print("Countries in MEP data missing from demography data:")
print(missing_in_demo)

# ------------------------------------------------------------
# Merge: each MEP receives their country's averaged values
# ------------------------------------------------------------

final_analysis_df = cleaned_df.merge(
    demography_avg,
    on="country_key",
    how="left",
    validate="many_to_one"
)

print("Cleaned MEP dataset:", cleaned_df.shape)
print("Final merged dataset:", final_analysis_df.shape)

# ------------------------------------------------------------
# Check merge coverage
# ------------------------------------------------------------

demo_added_cols = [
    c for c in demography_avg.columns
    if c != "country_key"
]

coverage = final_analysis_df[demo_added_cols].notna().any(axis=1).mean()
print(f"Merge coverage: {coverage:.1%}")

unmatched = final_analysis_df.loc[
    final_analysis_df[demo_added_cols].isna().all(axis=1),
    ["mep_name_clean", "Country", "country_clean_final", "country_key"]
].drop_duplicates()

print("Unmatched MEP countries:")
display(unmatched)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FINAL_ANALYSIS_PATH = TABLES_DIR / "final_analysis.csv"

final_analysis_df.to_csv(FINAL_ANALYSIS_PATH, index=False)

print(f"Saved final analysis dataset to: {FINAL_ANALYSIS_PATH}")

Country-level averaged demography: (28, 18)
Countries in MEP data missing from demography data:
[]
Cleaned MEP dataset: (193, 44)
Final merged dataset: (193, 61)
Merge coverage: 100.0%
Unmatched MEP countries:


,mep_name_clean,Country,country_clean_final,country_key


Saved final analysis dataset to: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\7 Final Analysis\outputs\tables\final_analysis.csv


In [4]:
# ============================================================
# Updated analysis block with country-level demography variables
# ============================================================

final_df = pd.read_csv(TABLES_DIR / "final_analysis.csv")

print(final_df.shape)
display(final_df.head())

(193, 61)


,mep_name_clean,Country,country_clean_final,dominant_topic,n_relevant_speeches,Ageing & Pensions,Borderline: Economic governance and employment,Borderline: Island regions and cohesion,Borderline: Labour market and social policy,Borderline: Regional development policy,...,employment_rate_55_64_avg_2010_2025,youth_unemployment_avg_2010_2025,debt_to_gdp_avg_2010_2025,age_started_receiving_old_age_pension_avg_2010_2025,continued_or_stopped_working_at_pension_start_avg_2010_2025,labour_force_participation_55_64_avg_2010_2025,old_age_poverty_rate_avg_2010_2025,projected_old_age_dependency_ratio_2050_avg_2010_2025,foreign_born_population_share_avg_2010_2025,demography_years_available_2010_2025
0,abir al sahlani,SW,se,Migration & Population Change,2,0.010289,0.000082,0.000082,0.000082,0.111688,...,75.11250,21.61875,38.816667,64.10,40.01,74.769375,15.521198,5.725300e+05,3.117294e+04,16
1,adam kosa,HU,hu,Other,2,0.200596,0.000285,0.000285,0.028850,0.000285,...,52.69375,16.60625,73.583333,61.50,40.01,56.374479,13.380156,5.328143e+05,3.542376e+04,16
2,agnes le brun,FR,fr,Other,1,0.001075,0.000215,0.000215,0.000215,0.391120,...,52.18750,22.40625,105.300000,60.70,39.98,53.625417,14.141129,3.685686e+06,1.292037e+06,16
3,agnieszka koz owska rajewicz,PL,pl,"Fertility, Families & Gender",1,0.000199,0.000040,0.000040,0.000040,0.102693,...,46.83750,16.97500,51.875000,59.95,40.00,51.627083,14.697379,2.062313e+06,8.114412e+05,16
4,alain cadec,FR,fr,Ageing & Pensions,1,0.542029,0.000339,0.000339,0.000339,0.326859,...,52.18750,22.40625,105.300000,60.70,39.98,53.625417,14.141129,3.685686e+06,1.292037e+06,16


In [5]:
# ------------------------------------------------------------
# Define outcome variables
# ------------------------------------------------------------

polarisation_vars = [
    "polarisation_score",
    "us_vs_them",
    "emotional_intensity",
    "moral_absolutism",
    "hostility_to_opponents"
]

topic_cols = [
    "Ageing & Pensions",
    "Borderline: Economic governance and employment",
    "Borderline: Island regions and cohesion",
    "Borderline: Labour market and social policy",
    "Borderline: Regional development policy",
    "Borderline: Youth / demographic future (weak signal)",
    "Fertility, Families & Gender",
    "Health & Care Services",
    "Housing & Living Conditions",
    "Migration & Population Change",
    "Population Growth & Sustainability",
    "Regional Depopulation & Territorial Cohesion",
    "Youth & Future Generations",
    "Other"
]

country_demo_cols = [
    "fertility_rate_avg_2010_2025",
    "old_age_dependency_ratio_avg_2010_2025",
    "median_age_avg_2010_2025",
    "life_expectancy_avg_2010_2025",
    "net_migration_avg_2010_2025",
    "population_change_avg_2010_2025",
    "gdp_per_capita_index_eu27_100_avg_2010_2025",
    "employment_rate_55_64_avg_2010_2025",
    "youth_unemployment_avg_2010_2025",
    "debt_to_gdp_avg_2010_2025",
    "age_started_receiving_old_age_pension_avg_2010_2025",
    "continued_or_stopped_working_at_pension_start_avg_2010_2025",
    "labour_force_participation_55_64_avg_2010_2025",
    "old_age_poverty_rate_avg_2010_2025",
    "projected_old_age_dependency_ratio_2050_avg_2010_2025",
    "foreign_born_population_share_avg_2010_2025",
    "demography_years_available_2010_2025"
]

mep_level_cols = [
    "age_2020",
    "gender",
    "political_group",
    "national_party",
    "country_clean_final",
    "confidence",
    "n_relevant_speeches"
]

In [6]:
# ------------------------------------------------------------
# Check missingness and distributions
# ------------------------------------------------------------

missing_demo = (
    final_df[country_demo_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("Missingness in country-level variables:")
display(missing_demo)

display(final_df[country_demo_cols].describe().T)

Missingness in country-level variables:


employment_rate_55_64_avg_2010_2025                            0.031088
projected_old_age_dependency_ratio_2050_avg_2010_2025          0.031088
youth_unemployment_avg_2010_2025                               0.031088
gdp_per_capita_index_eu27_100_avg_2010_2025                    0.031088
debt_to_gdp_avg_2010_2025                                      0.031088
age_started_receiving_old_age_pension_avg_2010_2025            0.031088
continued_or_stopped_working_at_pension_start_avg_2010_2025    0.031088
fertility_rate_avg_2010_2025                                   0.000000
old_age_dependency_ratio_avg_2010_2025                         0.000000
population_change_avg_2010_2025                                0.000000
median_age_avg_2010_2025                                       0.000000
life_expectancy_avg_2010_2025                                  0.000000
net_migration_avg_2010_2025                                    0.000000
labour_force_participation_55_64_avg_2010_2025                 0

,count,mean,std,min,25%,50%,75%,max
fertility_rate_avg_2010_2025,193.0,1.533057e+00,1.970309e-01,1.250667,1.341333,1.478667e+00,1.654000e+00,1.882667e+00
old_age_dependency_ratio_avg_2010_2025,193.0,2.471583e+01,9.494168e-01,22.249353,23.912994,2.454652e+01,2.573705e+01,2.605689e+01
median_age_avg_2010_2025,193.0,4.276245e+01,2.138266e+00,37.037500,41.381250,4.243750e+01,4.454667e+01,4.619375e+01
life_expectancy_avg_2010_2025,193.0,7.943244e+01,2.772346e+00,73.846667,76.986667,8.054000e+01,8.170667e+01,8.224000e+01
net_migration_avg_2010_2025,193.0,3.040007e+00,3.876511e+00,-3.283333,1.775000,1.925000e+00,5.433333e+00,2.449167e+01
population_change_avg_2010_2025,193.0,1.595707e+00,5.723488e+00,-9.350000,-2.383333,3.108333e+00,4.208333e+00,2.578333e+01
gdp_per_capita_index_eu27_100_avg_2010_2025,187.0,1.011667e+02,5.638746e+00,89.216688,96.096656,1.014776e+02,1.054033e+02,1.098589e+02
employment_rate_55_64_avg_2010_2025,187.0,5.319054e+01,8.780632e+00,39.456250,46.837500,5.144375e+01,5.883125e+01,7.511250e+01
youth_unemployment_avg_2010_2025,187.0,2.189878e+01,8.658952e+00,7.518750,16.975000,2.240625e+01,2.752812e+01,4.011250e+01
debt_to_gdp_avg_2010_2025,187.0,8.162175e+01,3.668910e+01,15.333333,51.875000,7.358333e+01,1.053000e+02,1.790667e+02


In [7]:
# ------------------------------------------------------------
# Country-level correlations with polarisation
# ------------------------------------------------------------

country_corr = (
    final_df[country_demo_cols + ["polarisation_score"]]
    .corr(numeric_only=True)["polarisation_score"]
    .drop("polarisation_score")
    .sort_values(ascending=False)
)

display(country_corr)

fertility_rate_avg_2010_2025                                   0.096714
projected_old_age_dependency_ratio_2050_avg_2010_2025          0.093560
life_expectancy_avg_2010_2025                                  0.080744
gdp_per_capita_index_eu27_100_avg_2010_2025                    0.073134
debt_to_gdp_avg_2010_2025                                      0.070591
foreign_born_population_share_avg_2010_2025                    0.065437
population_change_avg_2010_2025                                0.045766
age_started_receiving_old_age_pension_avg_2010_2025            0.033951
old_age_dependency_ratio_avg_2010_2025                         0.033944
employment_rate_55_64_avg_2010_2025                            0.021150
youth_unemployment_avg_2010_2025                               0.006685
net_migration_avg_2010_2025                                   -0.025336
labour_force_participation_55_64_avg_2010_2025                -0.047635
old_age_poverty_rate_avg_2010_2025                            -0

In [9]:
# ============================================================
# Self-contained full model block
# ============================================================

final_df = pd.read_csv(TABLES_DIR / "final_analysis.csv")

# ------------------------------------------------------------
# Outcomes
# ------------------------------------------------------------

target = "polarisation_score"

polarisation_vars = [
    "polarisation_score",
    "us_vs_them",
    "emotional_intensity",
    "moral_absolutism",
    "hostility_to_opponents"
]

# ------------------------------------------------------------
# Topic variables
# ------------------------------------------------------------

topic_cols = [
    "Ageing & Pensions",
    "Borderline: Economic governance and employment",
    "Borderline: Island regions and cohesion",
    "Borderline: Labour market and social policy",
    "Borderline: Regional development policy",
    "Borderline: Youth / demographic future (weak signal)",
    "Fertility, Families & Gender",
    "Health & Care Services",
    "Housing & Living Conditions",
    "Migration & Population Change",
    "Population Growth & Sustainability",
    "Regional Depopulation & Territorial Cohesion",
    "Youth & Future Generations",
    "Other"
]

topic_cols = [c for c in topic_cols if c in final_df.columns]

# ------------------------------------------------------------
# Country-level demographic variables
# ------------------------------------------------------------

country_demo_cols = [
    c for c in final_df.columns
    if c.endswith("_avg_2010_2025")
]

# optional, usually not a substantive predictor
country_demo_cols = [
    c for c in country_demo_cols
    if c != "demography_years_available_2010_2025"
]

# ------------------------------------------------------------
# MEP-level variables
# ------------------------------------------------------------

mep_level_candidates = [
    "age_2020",
    "gender",
    "political_group",
    "national_party",
    "country_clean_final",
    "confidence",
    "n_relevant_speeches"
]

base_cols = [c for c in mep_level_candidates if c in final_df.columns]

# ------------------------------------------------------------
# Build full model predictors
# ------------------------------------------------------------

full_cols = base_cols + country_demo_cols + topic_cols
full_cols = list(dict.fromkeys(full_cols))  # remove duplicates safely

print("MEP-level columns:")
print(base_cols)

print("\nCountry-level columns:")
print(country_demo_cols)

print("\nTopic columns:")
print(topic_cols)

print(f"\nNumber of predictors before encoding: {len(full_cols)}")

X_full = final_df[full_cols].copy()
y = final_df[target].copy()

# Drop rows with missing outcome
valid_rows = y.notna()
X_full = X_full.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

# ------------------------------------------------------------
# Preprocessing and model
# ------------------------------------------------------------

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

numeric_features = X_full.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_full.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))
])

preprocess_full = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

ridge_full = Pipeline([
    ("preprocess", preprocess_full),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 50)))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

# ------------------------------------------------------------
# Fit and evaluate
# ------------------------------------------------------------

full_r2 = cross_val_score(ridge_full, X_full, y, cv=cv, scoring="r2")
full_mae = -cross_val_score(ridge_full, X_full, y, cv=cv, scoring="neg_mean_absolute_error")

print("\nFull model")
print("CV R2:", full_r2.mean(), full_r2)
print("CV MAE:", full_mae.mean(), full_mae)

ridge_full.fit(X_full, y)

print("\nFull model fitted successfully.")

MEP-level columns:
['age_2020', 'country_clean_final', 'confidence', 'n_relevant_speeches']

Country-level columns:
['fertility_rate_avg_2010_2025', 'old_age_dependency_ratio_avg_2010_2025', 'median_age_avg_2010_2025', 'life_expectancy_avg_2010_2025', 'net_migration_avg_2010_2025', 'population_change_avg_2010_2025', 'gdp_per_capita_index_eu27_100_avg_2010_2025', 'employment_rate_55_64_avg_2010_2025', 'youth_unemployment_avg_2010_2025', 'debt_to_gdp_avg_2010_2025', 'age_started_receiving_old_age_pension_avg_2010_2025', 'continued_or_stopped_working_at_pension_start_avg_2010_2025', 'labour_force_participation_55_64_avg_2010_2025', 'old_age_poverty_rate_avg_2010_2025', 'projected_old_age_dependency_ratio_2050_avg_2010_2025', 'foreign_born_population_share_avg_2010_2025']

Topic columns:
['Ageing & Pensions', 'Borderline: Economic governance and employment', 'Borderline: Island regions and cohesion', 'Borderline: Labour market and social policy', 'Borderline: Regional development policy', 

In [10]:
# ============================================================
# Full model visualisations
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_predict
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")

# Fit full model
ridge_full.fit(X_full, y)

# Create predictions using cross-validation
y_pred_cv = cross_val_predict(ridge_full, X_full, y, cv=cv)

plot_df = final_df.copy()
plot_df["actual_polarisation"] = y
plot_df["predicted_polarisation"] = y_pred_cv
plot_df["residual"] = plot_df["actual_polarisation"] - plot_df["predicted_polarisation"]

In [12]:
# ============================================================
# Create safe plotting dataframe after model fitting
# ============================================================

from sklearn.model_selection import cross_val_predict
import numpy as np

# Cross-validated predictions
y_pred_cv = cross_val_predict(ridge_full, X_full, y, cv=cv)

plot_df = X_full.copy()
plot_df["actual_polarisation"] = y.values
plot_df["predicted_polarisation"] = y_pred_cv
plot_df["residual"] = plot_df["actual_polarisation"] - plot_df["predicted_polarisation"]

# Find a usable group column for colouring
possible_group_cols = [
    "political_group",
    "ep_group",
    "party_group",
    "group",
    "national_party",
    "country_clean_final"
]

hue_col = next((c for c in possible_group_cols if c in plot_df.columns), None)

print("Using hue column:", hue_col)
print("Available plot columns:")
print(plot_df.columns.tolist())

Using hue column: country_clean_final
Available plot columns:
['age_2020', 'country_clean_final', 'confidence', 'n_relevant_speeches', 'fertility_rate_avg_2010_2025', 'old_age_dependency_ratio_avg_2010_2025', 'median_age_avg_2010_2025', 'life_expectancy_avg_2010_2025', 'net_migration_avg_2010_2025', 'population_change_avg_2010_2025', 'gdp_per_capita_index_eu27_100_avg_2010_2025', 'employment_rate_55_64_avg_2010_2025', 'youth_unemployment_avg_2010_2025', 'debt_to_gdp_avg_2010_2025', 'age_started_receiving_old_age_pension_avg_2010_2025', 'continued_or_stopped_working_at_pension_start_avg_2010_2025', 'labour_force_participation_55_64_avg_2010_2025', 'old_age_poverty_rate_avg_2010_2025', 'projected_old_age_dependency_ratio_2050_avg_2010_2025', 'foreign_born_population_share_avg_2010_2025', 'Ageing & Pensions', 'Borderline: Economic governance and employment', 'Borderline: Island regions and cohesion', 'Borderline: Labour market and social policy', 'Borderline: Regional development policy',